# W15C1 Lab: Measuring Bias, Not Just Asserting It

Run every cell from the top. **Everything already works.**

Uses MiniLM, cached from Week 8. Everything here is measurement: the
argument in Stochastic Parrots is not that models are biased in the
abstract, it is that the bias is measurable and inherited from the data.

Today you will:

1. Measure an association in a real pretrained model, with a number.
2. Check whether that number could have happened by chance.
3. Try to remove the bias, and see how much you actually removed.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import itertools
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, logging
from sklearn.metrics.pairwise import cosine_similarity

logging.set_verbosity_error()
NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME)
model.eval()

@torch.no_grad()
def vec(word):
    ids = tok(word, return_tensors="pt")
    return model(**ids).last_hidden_state.mean(dim=1).numpy()[0]

print("model loaded:", NAME)

## Part 1. Put a number on it

The WEAT test: take two sets of target words and two sets of attribute
words, and ask whether the targets sit closer to one attribute set than
the other. It is cosine similarity, arranged carefully.

In [ ]:
# GIVEN. The association score, and one measurement.
def association(word, set_a, set_b):
    """How much closer is `word` to set_a than to set_b?"""
    v = vec(word).reshape(1, -1)
    a = np.mean([cosine_similarity(v, vec(w).reshape(1, -1))[0][0] for w in set_a])
    b = np.mean([cosine_similarity(v, vec(w).reshape(1, -1))[0][0] for w in set_b])
    return a - b

MALE = ["he", "man", "his", "father", "brother", "son"]
FEMALE = ["she", "woman", "her", "mother", "sister", "daughter"]

CAREERS = ["engineer", "programmer", "scientist", "surgeon"]
CARING = ["nurse", "teacher", "receptionist", "carer"]

rows = []
for job in CAREERS + CARING:
    rows.append({"word": job, "score": round(association(job, MALE, FEMALE), 4)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
print()
print("Positive means closer to the male words, negative means closer to female.")

plt.figure(figsize=(7, 3))
colours = ["#7C2529" if s > 0 else "#999" for s in table["score"]]
plt.bar(table["word"], table["score"], color=colours)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("female  <->  male"); plt.xticks(rotation=35, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 1 ==================
# Measure a different pairing. Pick two attribute sets and a group of
# words you expect to split between them.
#
# Suggestions: pleasant/unpleasant words against two sets of names;
# or young/old words against job titles.
#
# Expected: you will usually find some separation. The important discipline is
#           that you decided what to measure BEFORE looking, otherwise you are
#           hunting through combinations until one looks bad, which proves nothing.
# ===============================================
SET_A = ["young", "new", "fresh", "modern"]
SET_B = ["old", "aged", "elderly", "outdated"]
WORDS = ["engineer", "nurse", "professor", "intern", "manager"]

for w in WORDS:
    print(f"   {w:<12} {association(w, SET_A, SET_B):+.4f}")

## Part 2. Could it be chance?

A nonzero number is not evidence on its own. Shuffle the labels many times
and see how often pure chance produces a gap this big.

In [ ]:
# GIVEN. A permutation test, which is the honest version of the measurement.
observed = np.mean([association(w, MALE, FEMALE) for w in CAREERS]) - \
           np.mean([association(w, MALE, FEMALE) for w in CARING])

pool = CAREERS + CARING
rng = np.random.default_rng(0)
scores = {w: association(w, MALE, FEMALE) for w in pool}

null = []
for _ in range(200):
    shuffled = rng.permutation(pool)
    half = len(pool) // 2
    null.append(np.mean([scores[w] for w in shuffled[:half]]) -
                np.mean([scores[w] for w in shuffled[half:]]))

p_value = np.mean([abs(n) >= abs(observed) for n in null])
print(f"observed gap: {observed:+.4f}")
print(f"p-value     : {p_value:.3f}  (how often chance alone beats it)")

plt.figure(figsize=(6.5, 3))
plt.hist(null, bins=30, color="#999")
plt.axvline(observed, color="#7C2529", linewidth=2, label="observed")
plt.xlabel("gap under random relabelling"); plt.legend()
plt.title(f"p = {p_value:.3f}"); plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# Eight words is a very small sample. Add more job titles to both lists
# and see what happens to the p-value.
#
# Expected: p-values move a lot with small word lists, and the honest reading of
#           a large p is 'this test cannot tell'. Published bias results use much
#           longer lists for exactly this reason, and a single striking word pair
#           is never evidence on its own.
# ===============================================
MORE_CAREERS = ["engineer", "programmer", "scientist", "surgeon"]   # <-- add to these
MORE_CARING = ["nurse", "teacher", "receptionist", "carer"]

pool2 = MORE_CAREERS + MORE_CARING
scores2 = {w: association(w, MALE, FEMALE) for w in pool2}
obs2 = (np.mean([scores2[w] for w in MORE_CAREERS])
        - np.mean([scores2[w] for w in MORE_CARING]))

rng2 = np.random.default_rng(0)
null2 = []
for _ in range(200):
    s = rng2.permutation(pool2); h = len(pool2) // 2
    null2.append(np.mean([scores2[w] for w in s[:h]]) - np.mean([scores2[w] for w in s[h:]]))

print(f"words in test: {len(pool2)}")
print(f"observed gap : {obs2:+.4f}")
print(f"p-value      : {np.mean([abs(n) >= abs(obs2) for n in null2]):.3f}")

## Part 3. Try to remove it

The obvious fix is to find the direction in the space that encodes the
attribute and project it away. It works, partly, and the part it misses is
the whole debate.

In [ ]:
# GIVEN. Project out the gender direction and measure again.
direction = np.mean([vec(w) for w in MALE], axis=0) - np.mean([vec(w) for w in FEMALE], axis=0)
direction = direction / np.linalg.norm(direction)

def debiased(word):
    v = vec(word)
    return v - np.dot(v, direction) * direction          # remove that component

def association_debiased(word, set_a, set_b):
    v = debiased(word).reshape(1, -1)
    a = np.mean([cosine_similarity(v, debiased(w).reshape(1, -1))[0][0] for w in set_a])
    b = np.mean([cosine_similarity(v, debiased(w).reshape(1, -1))[0][0] for w in set_b])
    return a - b

rows = []
for job in CAREERS + CARING:
    rows.append({"word": job,
                 "before": round(association(job, MALE, FEMALE), 4),
                 "after": round(association_debiased(job, MALE, FEMALE), 4)})
out = pd.DataFrame(rows)
print(out.to_string(index=False))

x = np.arange(len(out))
plt.figure(figsize=(8, 3))
plt.bar(x - 0.2, out["before"], 0.4, label="before", color="#7C2529")
plt.bar(x + 0.2, out["after"], 0.4, label="after", color="#999")
plt.xticks(x, out["word"], rotation=35, ha="right"); plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("female <-> male"); plt.legend(); plt.tight_layout(); plt.show()

print(f"\nmean absolute score before: {out['before'].abs().mean():.4f}")
print(f"mean absolute score after : {out['after'].abs().mean():.4f}")

In [ ]:
# ================== YOUR TURN 3 ==================
# Debiasing removed one direction. Check whether the WORDS still cluster
# by the old groups, using a similarity the debiasing never touched.
#
# Expected: the direct male/female score collapses, but the job words often still
#           sit closer to their own group than to the other. Removing a direction
#           hides the measurement you aimed at without removing the structure
#           underneath, which is the central criticism of geometric debiasing.
# ===============================================
def group_cohesion(words, embed):
    pairs = list(itertools.combinations(words, 2))
    return np.mean([cosine_similarity(embed(a).reshape(1, -1),
                                      embed(b).reshape(1, -1))[0][0] for a, b in pairs])

for label, embed in (("original", vec), ("debiased", debiased)):
    within = (group_cohesion(CAREERS, embed) + group_cohesion(CARING, embed)) / 2
    across = np.mean([cosine_similarity(embed(a).reshape(1, -1), embed(b).reshape(1, -1))[0][0]
                      for a in CAREERS for b in CARING])
    print(f"   {label:<10} within-group {within:.3f}   across-group {across:.3f}   "
          f"gap {within - across:+.3f}")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Most attribute pairings show some separation. The discipline that makes it
#   evidence rather than anecdote is choosing the word lists before you look at
#   the scores. Searching combinations until one looks bad is how you find a
#   result in pure noise.
#
# YOUR TURN 2
#   With eight words the p-value is unstable; adding words usually moves it a
#   lot. A large p means "this test cannot tell", not "there is no bias".
#   Published WEAT studies use lists of dozens of words for this reason.
#
# YOUR TURN 3
#   The direct score collapses while the group structure largely survives. That
#   is the standard criticism of projection-based debiasing, made concrete: you
#   removed the coordinate your metric was reading, not the information. A
#   downstream model can still recover it from what remains.